# 🌋 Elemental Genesis — Text-to-Video Baseline Generation
## Selected Topics in AI 2 — Phase 1

This notebook generates baseline videos using **ModelScope T2V** and runs the weakness analysis.

**Instructions**: Runtime → Change runtime type → **T4 GPU** → Run all cells

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg opencv-python safetensors
!pip install -q lpips open-clip-torch einops scikit-image
print('All dependencies installed!')

In [ ]:
# Cell 2: Define Prompts

FIRE_PROMPTS = [
    "A blazing campfire in a dark forest with sparks flying upward into the night sky",
    "A volcanic eruption with glowing lava streams flowing down a mountainside",
    "A phoenix rising from golden flames against a starry night sky",
    "Molten lava flowing slowly through a rocky canyon, glowing orange and red",
]

WATER_PROMPTS = [
    "Ocean waves crashing dramatically on rocky shores at golden sunset",
    "A gentle waterfall cascading into a crystal clear turquoise pool in a jungle",
    "Rain drops falling on a calm lake surface creating expanding ripples",
    "An underwater scene with colorful fish swimming through vibrant coral reefs",
]

EARTH_PROMPTS = [
    "Sand dunes shifting slowly in a vast desert under golden hour light",
    "Crystals growing rapidly from the ground inside a dark glowing cave",
    "A massive landslide cascading down a forested mountain slope",
    "Tectonic plates splitting the ground apart in a barren desert landscape",
]

WIND_PROMPTS = [
    "A powerful tornado forming over an open golden wheat field",
    "Autumn leaves swirling in a strong gust of wind through a forest path",
    "A sandstorm approaching a small desert village at dusk",
    "Dramatic clouds moving rapidly across a colorful sunset sky",
]

SIMPLE_PROMPTS = [
    "A burning candle on a table",
    "Ocean waves on a beach",
    "A tree in the wind",
    "Clouds moving in the sky",
]

COMPLEX_PROMPTS = [
    "A red bird flying over a blue ocean while a volcano erupts in the background",
    "Three dolphins jumping out of the water simultaneously at sunset",
    "A tornado made of fire spinning through an icy glacier landscape",
    "A waterfall flowing upward into the sky while leaves fall downward around it",
]

ALL_PROMPTS = {
    "fire": FIRE_PROMPTS,
    "water": WATER_PROMPTS,
    "earth": EARTH_PROMPTS,
    "wind": WIND_PROMPTS,
    "simple": SIMPLE_PROMPTS,
    "complex": COMPLEX_PROMPTS,
}

print(f'Total prompts: {sum(len(v) for v in ALL_PROMPTS.values())}')

In [ ]:
# Cell 3: Load ModelScope T2V Pipeline
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print('Pipeline loaded successfully!')
print(f'Device: {pipe.device}')

In [ ]:
# Cell 4: Generate ALL videos
import os
import numpy as np
import imageio
from PIL import Image

OUTPUT_DIR = '/content/outputs/baseline'

def generate_and_save(pipe, prompt, save_path, num_frames=16, seed=42):
    generator = torch.Generator(device='cpu').manual_seed(seed)
    output = pipe(
        prompt=prompt,
        num_frames=num_frames,
        height=256, width=256,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator,
    )
    frames = output.frames[0]
    
    # Save video
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    video_array = [np.array(f) for f in frames]
    imageio.mimsave(save_path, video_array, fps=8, codec='libx264')
    
    # Save frames
    frame_dir = save_path.replace('.mp4', '_frames')
    os.makedirs(frame_dir, exist_ok=True)
    for j, frame in enumerate(frames):
        frame.save(os.path.join(frame_dir, f'frame_{j:03d}.png'))
    
    return frames

all_frames = {}
for category, prompts in ALL_PROMPTS.items():
    print(f'\n=== {category.upper()} ===')
    all_frames[category] = []
    for i, prompt in enumerate(prompts):
        print(f'  [{i+1}/{len(prompts)}] {prompt[:60]}...')
        path = os.path.join(OUTPUT_DIR, category, f'{category}_{i+1:02d}.mp4')
        frames = generate_and_save(pipe, prompt, path)
        all_frames[category].append(frames)
        print(f'    Saved: {path}')

print('\n All videos generated!')

In [ ]:
# Cell 5: Display sample videos as GIFs
from IPython.display import display, HTML
import matplotlib.pyplot as plt

def show_video_grid(frames_list, titles, cols=4):
    """Display first frame of each video in a grid."""
    rows = (len(frames_list) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
    for i, (frames, title) in enumerate(zip(frames_list, titles)):
        axes[i].imshow(np.array(frames[0]))
        axes[i].set_title(title[:30], fontsize=9)
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

# Show first frames from each element
for cat in ['fire', 'water', 'earth', 'wind']:
    print(f'\n{cat.upper()} samples:')
    show_video_grid(
        all_frames[cat],
        ALL_PROMPTS[cat]
    )

In [ ]:
# Cell 6: Evaluate Metrics — CLIP-SIM
import open_clip
from torchvision import transforms

# Load CLIP
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.cuda().eval()

@torch.no_grad()
def compute_clip_sim(frames, prompt):
    text_tokens = clip_tokenizer([prompt]).cuda()
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
    
    sims = []
    for frame in frames:
        img = clip_preprocess(frame).unsqueeze(0).cuda()
        img_feat = clip_model.encode_image(img)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        sims.append((img_feat @ text_feat.T).item())
    return np.mean(sims), sims

print('CLIP-SIM Results:')
print(f'{"Category":<12} {"Mean CLIP-SIM":>14}')
print('-'*30)
clip_results = {}
for cat, prompts in ALL_PROMPTS.items():
    scores = []
    for i, prompt in enumerate(prompts):
        if cat in all_frames and i < len(all_frames[cat]):
            score, _ = compute_clip_sim(all_frames[cat][i], prompt)
            scores.append(score)
    if scores:
        clip_results[cat] = np.mean(scores)
        print(f'{cat:<12} {np.mean(scores):>14.4f}')

In [ ]:
# Cell 7: Evaluate Metrics — Temporal LPIPS
import lpips

lpips_fn = lpips.LPIPS(net='alex').cuda().eval()
to_tensor = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

@torch.no_grad()
def compute_temporal_lpips(frames):
    dists = []
    for i in range(len(frames) - 1):
        img1 = to_tensor(frames[i]).unsqueeze(0).cuda()
        img2 = to_tensor(frames[i+1]).unsqueeze(0).cuda()
        d = lpips_fn(img1, img2).item()
        dists.append(d)
    return np.mean(dists), dists

print('Temporal LPIPS Results (lower = more consistent):')
print(f'{"Category":<12} {"Mean T-LPIPS":>14}')
print('-'*30)
lpips_results = {}
for cat in ALL_PROMPTS:
    scores = []
    for i in range(len(ALL_PROMPTS[cat])):
        if cat in all_frames and i < len(all_frames[cat]):
            score, _ = compute_temporal_lpips(all_frames[cat][i])
            scores.append(score)
    if scores:
        lpips_results[cat] = np.mean(scores)
        print(f'{cat:<12} {np.mean(scores):>14.4f}')

In [ ]:
# Cell 8: Summary Table — Weakness Analysis
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

def compute_ssim_psnr(frames):
    ssim_vals, psnr_vals = [], []
    for i in range(len(frames) - 1):
        f1, f2 = np.array(frames[i]), np.array(frames[i+1])
        ssim_vals.append(structural_similarity(f1, f2, channel_axis=2, data_range=255))
        psnr_vals.append(peak_signal_noise_ratio(f1, f2, data_range=255))
    return np.mean(ssim_vals), np.mean(psnr_vals)

print('\n' + '='*70)
print('COMPLETE WEAKNESS ANALYSIS RESULTS')
print('='*70)
print(f'{"Category":<12} {"CLIP-SIM":>10} {"T-LPIPS":>10} {"SSIM":>8} {"PSNR":>8}')
print('-'*52)

for cat in ALL_PROMPTS:
    clip_s = clip_results.get(cat, 0)
    lpips_s = lpips_results.get(cat, 0)
    ssim_scores, psnr_scores = [], []
    for i in range(len(ALL_PROMPTS[cat])):
        if cat in all_frames and i < len(all_frames[cat]):
            s, p = compute_ssim_psnr(all_frames[cat][i])
            ssim_scores.append(s)
            psnr_scores.append(p)
    avg_ssim = np.mean(ssim_scores) if ssim_scores else 0
    avg_psnr = np.mean(psnr_scores) if psnr_scores else 0
    print(f'{cat:<12} {clip_s:>10.4f} {lpips_s:>10.4f} {avg_ssim:>8.4f} {avg_psnr:>8.2f}')

print('='*52)
print('\nWeakness 1 (Temporal Flickering): fire & wind have higher T-LPIPS')
print('Weakness 2 (Semantic Misalignment): complex prompts have lower CLIP-SIM')

In [ ]:
# Cell 9: Download all outputs
import shutil
shutil.make_archive('/content/elemental_genesis_outputs', 'zip', '/content/outputs')
from google.colab import files
files.download('/content/elemental_genesis_outputs.zip')
print('Download started!')